<a href="https://colab.research.google.com/github/ShlegelMamedov/strukturapython/blob/main/krutie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🍽️ Система заказов кафе
# Авторы: Шлегель Матвей, Мамедов Рамин | Группа 2-ИС33

def create_cafe_system():
    """Создаёт систему управления заказами кафе."""

    menu = {}  # {название: {цена, категория, время}}
    orders = {}  # {id_заказа: {стол, блюда, статус, сумма}}
    order_counter = 1

    # 1. Добавление блюда в меню
    def add_dish(name, price, category, prep_time):
        """Добавить блюдо в меню: название, цена, категория, время приготовления"""
        menu[name] = {
            "price": price,
            "category": category,
            "prep_time": prep_time
        }
        print(f"✅ '{name}' добавлено в меню | {price}₽ | {category}")

    # 2. Создание заказа
    def create_order(table, dishes):
        """Создать заказ: стол, список блюд, статус"""
        nonlocal order_counter
        # Проверка наличия блюд в меню
        valid_dishes = [d for d in dishes if d in menu]
        if not valid_dishes:
            print("❌ Нет доступных блюд в заказе")
            return None
        # Расчёт стоимости
        total = sum(menu[d]["price"] for d in valid_dishes)
        orders[order_counter] = {
            "table": table,
            "dishes": valid_dishes,
            "status": "в работе",
            "total": total
        }
        print(f"📋 Заказ #{order_counter} | Стол {table} | {len(valid_dishes)} блюд | {total}₽")
        order_counter += 1
        return order_counter - 1

    # 3. Расчёт стоимости заказа
    def calculate_order_total(order_id):
        """Вернуть стоимость заказа по ID"""
        if order_id in orders:
            return orders[order_id]["total"]
        return 0

    # 4. Самые популярные блюда
    def get_popular_dishes(top_n=3):
        """Найти top_n самых заказываемых блюд"""
        dish_count = {}
        for order in orders.values():
            for dish in order["dishes"]:
                dish_count[dish] = dish_count.get(dish, 0) + 1
        return sorted(dish_count.items(), key=lambda x: x[1], reverse=True)[:top_n]

    # 5. Выручка по категориям
    def get_revenue_by_category():
        """Статистика выручки по категориям блюд"""
        revenue = {}
        for order in orders.values():
            if order["status"] == "оплачен":  # только оплаченные заказы
                for dish in order["dishes"]:
                    if dish in menu:
                        cat = menu[dish]["category"]
                        revenue[cat] = revenue.get(cat, 0) + menu[dish]["price"]
        return revenue

    # Возвращаем все функции как словарь
    return {
        "add_dish": add_dish,
        "create_order": create_order,
        "calculate_order_total": calculate_order_total,
        "get_popular_dishes": get_popular_dishes,
        "get_revenue_by_category": get_revenue_by_category,
        "menu": menu,
        "orders": orders
    }

# === Тестирование системы ===
print("🍽️ Система кафе запущена!\n")
cafe = create_cafe_system()

# Добавляем блюда в меню
cafe["add_dish"]("Борщ", 350, "Супы", 15)
cafe["add_dish"]("Цезарь", 450, "Салаты", 10)
cafe["add_dish"]("Пицца", 600, "Горячее", 20)
cafe["add_dish"]("Кофе", 150, "Напитки", 5)
cafe["add_dish"]("Чизкейк", 300, "Десерты", 5)

# Создаём заказы
print("\n📝 Создаём заказы:")
cafe["create_order"](1, ["Борщ", "Кофе"])
cafe["create_order"](2, ["Пицца", "Цезарь", "Кофе"])
cafe["create_order"](1, ["Чизкейк", "Кофе"])
cafe["create_order"](3, ["Борщ", "Пицца"])

# Обновляем статусы для статистики
for oid in cafe["orders"]:
    cafe["orders"][oid]["status"] = "оплачен"

# Тестируем функции
print(f"\n💰 Стоимость заказа #1: {cafe['calculate_order_total'](1)}₽")
print(f"🔥 Популярные блюда: {cafe['get_popular_dishes']()}")
print(f"📊 Выручка по категориям: {cafe['get_revenue_by_category']()}")

In [ ]:
# 🎓 Система учёта успеваемости
# Авторы: Шлегель Матвей, Мамедов Рамин | Группа 2-ИС33

def create_gradebook_system():
    """Создаёт систему учёта успеваемости студентов."""

    students = {}  # {ID: {ФИО, группа, контакты, оценки}}

    # 1. Добавление студента
    def add_student(sid, name, group, contacts):
        """Добавить студента: ID, ФИО, группа, контакты"""
        students[sid] = {
            "name": name,
            "group": group,
            "contacts": contacts,
            "grades": []  # список оценок: {предмет, оценка, дата}
        }
        print(f"✅ Студент {name} ({group}) добавлен")

    # 2. Добавление оценки
    def add_grade(sid, subject, grade, date):
        """Добавить оценку: студент, предмет, оценка, дата"""
        if sid not in students:
            print(f"❌ Студент #{sid} не найден")
            return False
        if not (2 <= grade <= 5):
            print(f"❌ Недопустимая оценка: {grade}")
            return False
        students[sid]["grades"].append({
            "subject": subject,
            "grade": grade,
            "date": date
        })
        print(f"📝 {students[sid]['name']}: {subject} = {grade} ({date})")
        return True

    # 3. Средний балл студента
    def get_average_grade(sid):
        """Рассчитать средний балл студента"""
        if sid not in students or not students[sid]["grades"]:
            return 0
        grades = [g["grade"] for g in students[sid]["grades"]]
        return round(sum(grades) / len(grades), 2)

    # 4. Студенты с оценкой ниже порога
    def find_low_performers(threshold=3.0):
        """Найти студентов со средним баллом ниже threshold"""
        low_performers = []
        for sid, info in students.items():
            avg = get_average_grade(sid)
            if avg < threshold and info["grades"]:  # только с оценками
                low_performers.append({
                    "id": sid,
                    "name": info["name"],
                    "group": info["group"],
                    "average": avg
                })
        return sorted(low_performers, key=lambda x: x["average"])

    # 5. Рейтинг студентов
    def get_student_ranking():
        """Вернуть рейтинг студентов по среднему баллу"""
        ranking = []
        for sid, info in students.items():
            if info["grades"]:  # только с оценками
                ranking.append({
                    "id": sid,
                    "name": info["name"],
                    "group": info["group"],
                    "average": get_average_grade(sid)
                })
        return sorted(ranking, key=lambda x: x["average"], reverse=True)

    return {
        "add_student": add_student,
        "add_grade": add_grade,
        "get_average_grade": get_average_grade,
        "find_low_performers": find_low_performers,
        "get_student_ranking": get_student_ranking,
        "students": students
    }

# === Тестирование системы ===
print("🎓 Система успеваемости запущена!\n")
gradebook = create_gradebook_system()

# Добавляем студентов
gradebook["add_student"](1, "Шлегель Матвей", "2-ИС33", "matvey@email.com")
gradebook["add_student"](2, "Мамедов Рамин", "2-ИС33", "ramin@email.com")
gradebook["add_student"](3, "Иванова Анна", "2-ИС33", "anna@email.com")
gradebook["add_student"](4, "Петров Иван", "2-ИС32", "ivan@email.com")

# Добавляем оценки
print("\n📊 Добавляем оценки:")
gradebook["add_grade"](1, "Python", 5, "2026-04-01")
gradebook["add_grade"](1, "Web-dev", 4, "2026-04-05")
gradebook["add_grade"](1, "БД", 5, "2026-04-10")

gradebook["add_grade"](2, "Python", 5, "2026-04-01")
gradebook["add_grade"](2, "Web-dev", 5, "2026-04-05")
gradebook["add_grade"](2, "БД", 4, "2026-04-10")

gradebook["add_grade"](3, "Python", 4, "2026-04-01")
gradebook["add_grade"](3, "Web-dev", 3, "2026-04-05")

gradebook["add_grade"](4, "Python", 3, "2026-04-01")
gradebook["add_grade"](4, "Web-dev", 2, "2026-04-05")

# Тестируем функции
print(f"\n📈 Средние баллы:")
for sid in [1, 2, 3, 4]:
    name = gradebook["students"][sid]["name"]
    avg = gradebook["get_average_grade"](sid)
    print(f"   {name}: {avg}")

print(f"\n🔻 Студенты с баллом < 4.0:")
for s in gradebook["find_low_performers"](4.0):
    print(f"   {s['name']} ({s['group']}): {s['average']}")

print(f"\n🏆 Рейтинг студентов:")
for i, s in enumerate(gradebook["get_student_ranking"], 1):
    print(f"   {i}. {s['name']}: {s['average']}")